In [ ]:
!pip install annoy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.5/647.5 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for annoy: filename=annoy-1.17.3-cp312-cp312-linux_x86_64.whl size=551516 sha256=a74bac2473e1f178f9b06ae17f01c41ee1151e7a54db5c1a53a668d9415cf5b7
  Stored in directory: /root/.cache/pip/wheels/db/b9/53/a3b2d1fe1743abadddec6aa541294b24fdbc39d7800bc57311
Successfully built annoy


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
import numpy as np
import annoy
import os

In [ ]:
base_model = ResNet50(weights="imagenet", include_top=False)
x = base_model.output
x = GlobalAveragePooling2D()(x)
model = Model(inputs=base_model.input, outputs=x)
mode=model.trainable = False

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

In [ ]:
trainImage = ImageDataGenerator(rescale=1./255).flow_from_directory(trainPath,target_size=(224,224),batch_size=32,class_mode='categorical')
valiImage = ImageDataGenerator(rescale=1./255).flow_from_directory(valiPath,target_size=(224,224),batch_size=32,class_mode='categorical')

Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


In [ ]:
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

trainImage = datagen.flow_from_directory(
    trainPath, target_size=(224,224), class_mode=None, shuffle=False)

valiImage = datagen.flow_from_directory(
    valiPath, target_size=(224,224), class_mode=None, shuffle=False)

class_mode=None:
ค่า default: class_mode='categorical' สำหรับการจำแนกประเภท
class_mode=None หมายความว่าเราไม่ได้ใช้ป้ายกำกับ (labels) ในชุดข้อมูลนั้นๆ ซึ่งใช้กรณีที่เราไม่ต้องการข้อมูลป้ายกำกับ (เช่น ในการทดสอบหรือสร้าง feature extractor)

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import ImageFile

# Allow truncated images to be loaded
ImageFile.LOAD_TRUNCATED_IMAGES = True


# === โหลด ResNet50 ===
base_model = ResNet50(weights="imagenet", include_top=False)
x = base_model.output
x = GlobalAveragePooling2D()(x)   # ทำให้ได้ vector 2048
model = Model(inputs=base_model.input, outputs=x)

for layer in base_model.layers:
    layer.trainable = False

# === Generator ===
from google.colab import drive
drive.mount('/content/gdrive')

trainPath='/content/gdrive/MyDrive/train'
valiPath='/content/gdrive/MyDrive/val'

datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

trainImage = datagen.flow_from_directory(
    trainPath, target_size=(224,224), batch_size=32, class_mode=None, shuffle=False)

valiImage = datagen.flow_from_directory(
    valiPath, target_size=(224,224), batch_size=32, class_mode=None, shuffle=False)

# === Extract features ===
train_features = model.predict(trainImage, verbose=1)
val_features = model.predict(valiImage, verbose=1)

# === เก็บชื่อไฟล์กับ vector ===
train_filenames = trainImage.filenames
val_filenames = valiImage.filenames

df_train = pd.DataFrame(train_features)
df_train.insert(0, "filename", train_filenames)

df_val = pd.DataFrame(val_features)
df_val.insert(0, "filename", val_filenames)

# === Save CSV ===
df_train.to_csv("train_vectors.csv", index=False)
df_val.to_csv("val_vectors.csv", index=False)

print("✅ Saved feature vectors to train_vectors.csv and val_vectors.csv")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Found 2872 images belonging to 18 classes.
Found 761 images belonging to 18 classes.


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


90/90 ━━━━━━━━━━━━━━━━━━━━ 829s 9s/step
24/24 ━━━━━━━━━━━━━━━━━━━━ 192s 8s/step
✅ Saved feature vectors to train_vectors.csv and val_vectors.csv
